In [1]:
from google.colab import drive
drive.mount("/content/drive")

CKPT_BEST = "/content/drive/MyDrive/codegen_cp/checkpoints/best"

Mounted at /content/drive


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(CKPT_BEST)
model = AutoModelForCausalLM.from_pretrained(
    CKPT_BEST,
    torch_dtype=torch.bfloat16,
    device_map="auto",          # auto-places on GPU
)
model.eval()
print("Model loaded")
print(f"Parameters : {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")
print(f"GPU RAM    : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

Model loaded
Parameters : 354.9 M
GPU RAM    : 0.71 GB


In [3]:
def generate(prompt: str,
             max_new_tokens: int = 200,
             temperature: float = 0.2,
             top_p: float = 0.95,
             do_sample: bool = True) -> str:
    """Generate a code completion for the given prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [4]:
test_prompts = [
    # ── Classic algorithms ───────────────────────────────────
    "def binary_search(arr, target):\n    \"\"\"Return index of target in sorted arr, or -1 if not found.\"\"\"\n",
    "def merge_sort(arr):\n    \"\"\"Sort array using merge sort. Return sorted array.\"\"\"\n",

    # ── Competitive-programming patterns ─────────────────────
    "def solve():\n    n = int(input())\n    a = list(map(int, input().split()))\n    # find maximum subarray sum\n",
    "def count_inversions(arr):\n    \"\"\"Count number of inversions in array using merge sort.\"\"\"\n",

    # ── Graph / DP ───────────────────────────────────────────
    "def dijkstra(graph, src):\n    \"\"\"Shortest path from src using Dijkstra. graph is adjacency list.\"\"\"\n",
    "def knapsack(weights, values, capacity):\n    \"\"\"0/1 knapsack. Return max value.\"\"\"\n",
]

print("=" * 60)
for i, prompt in enumerate(test_prompts, 1):
    print(f"\n── Test {i} ──────────────────────────────────────────────")
    print("PROMPT:")
    print(prompt)
    print("COMPLETION:")
    result = generate(prompt, max_new_tokens=150, temperature=0.2)
    print(result)
    print("=" * 60)


── Test 1 ──────────────────────────────────────────────
PROMPT:
def binary_search(arr, target):
    """Return index of target in sorted arr, or -1 if not found."""

COMPLETION:
    low = 0
    high = len(arr) - 1
    while low <= high:
        mid = (low + high) // 2
        if arr[mid] < target:
            low = mid + 1
        elif arr[mid] > target:
            high = mid - 1
        else:
            return mid
    return -1

def main():
    arr = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    print binary_search(arr, 5)
    print binary_search(arr, 4)
    print binary_search(arr, 3)
    print binary_search(

── Test 2 ──────────────────────────────────────────────
PROMPT:
def merge_sort(arr):
    """Sort array using merge sort. Return sorted array."""

COMPLETION:
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = arr[:mid]
    right = arr[mid:]

    return merge_sort(left) + [right] + merge_sort(right)


def main():
    """Main program."""
    print("Array is:")

In [5]:
MY_PROMPT = """
def two_sum(nums, target):
    \"\"\"Return indices of two numbers that add up to target.\"\"\"
"""

print("PROMPT:", MY_PROMPT)
print("\nCOMPLETION:")
print(generate(MY_PROMPT, max_new_tokens=200, temperature=0.2))

PROMPT: 
def two_sum(nums, target):
    """Return indices of two numbers that add up to target."""


COMPLETION:
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return i, j
    return None


def sum_of_three(nums):
    """Return sum of three numbers in nums."""
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            for k in range(j + 1, len(nums)):
                if nums[i] + nums[j] + nums[k] == target:
                    return nums[i], nums[j], nums[k]
    return None


def sum_of_four(nums):
    """Return sum of four numbers in nums."""
    


In [6]:
import math

def compute_perplexity(code: str) -> float:
    """Compute perplexity of the model on a given code string."""
    enc = tokenizer(code, return_tensors="pt").to(device)
    input_ids = enc["input_ids"]

    with torch.no_grad():
        outputs = model(**enc, labels=input_ids)

    loss = outputs.loss.item()
    return math.exp(loss)


reference_snippets = {
    "binary_search": """
def binary_search(arr, target):
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target: return mid
        elif arr[mid] < target: lo = mid + 1
        else: hi = mid - 1
    return -1
""",
    "prefix_sum": """
def prefix_sum(arr):
    ps = [0] * (len(arr) + 1)
    for i, x in enumerate(arr):
        ps[i+1] = ps[i] + x
    return ps
""",
    "gcd": """
def gcd(a, b):
    while b:
        a, b = b, a % b
    return a
""",
}

print("\n── Perplexity scores (lower = better) ──────────────────")
for name, code in reference_snippets.items():
    ppl = compute_perplexity(code)
    print(f"  {name:<20} perplexity: {ppl:.2f}")



── Perplexity scores (lower = better) ──────────────────
  binary_search        perplexity: 1.48
  prefix_sum           perplexity: 2.15
  gcd                  perplexity: 1.70


In [8]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [9]:
BASE_MODEL_ID = "Salesforce/codegen-350M-mono"

print("Loading base model for comparison...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
base_model.eval()

def compute_perplexity_for(mdl, code: str) -> float:
    enc = tokenizer(code, return_tensors="pt").to(device)
    input_ids = enc["input_ids"]
    with torch.no_grad():
        outputs = mdl(**enc, labels=input_ids)
    return math.exp(outputs.loss.item())

print(f"\n{'Snippet':<22} {'Base PPL':>10} {'Fine-tuned PPL':>15} {'Improvement':>12}")
print("-" * 62)
for name, code in reference_snippets.items():
    base_ppl  = compute_perplexity_for(base_model, code)
    tuned_ppl = compute_perplexity_for(model, code)
    delta     = base_ppl - tuned_ppl
    arrow     = "✓ better" if delta > 0 else "✗ worse"
    print(f"{name:<22} {base_ppl:>10.2f} {tuned_ppl:>15.2f} {arrow:>12}")

Loading base model for comparison...


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Snippet                  Base PPL  Fine-tuned PPL  Improvement
--------------------------------------------------------------
binary_search                1.47            1.48      ✗ worse
prefix_sum                   2.18            2.15     ✓ better
gcd                          1.79            1.70     ✓ better
